# Parse Gesund.Bund ICD-10 Pages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys
sys.path.append((root_dir / 'crawl').as_posix())
sys.path.append((root_dir / 'search').as_posix())

parse html

In [ ]:
import json
import unicodedata
import pandas as pd
from tqdm.auto import tqdm
from gesund_bund import parse_icd10_html

def normalize(s):
    return unicodedata.normalize('NFKD', s).strip()

def parse_icd10_pages(tsv_file):
    records = []
    for path in tqdm((root_dir / 'data/gesund_bund').glob('*icd-code-*.json')):
        with path.open('r') as f:
            d = json.load(f)
            
        metadata, paragraphs = parse_icd10_html(d['html_content'])
        assert len(paragraphs) <= 1
        if 'title' in metadata:
            records.append({
                "id": metadata['icd10'].lower(),
                "url": d['url'],
                "date_crawled": d["time_crawled"].split('T')[0],
                "language": "en" if "/en/" in d['url'] else "de",
                "title": normalize(metadata['title']),
                "summary": normalize(paragraphs[0]['title']) if len(paragraphs) > 0 else None,
                "text": normalize(paragraphs[0]['text']) if len(paragraphs) > 0 else None,
            })

    df = pd.DataFrame.from_records(records).sort_values(by=['language', 'id'])
    df.to_csv(tsv_file, sep="\t", index=False)

In [ ]:
parse_icd10_pages(root_dir / 'data/gesund_bund_icd10.tsv')

In [ ]:
import pandas as pd

df = pd.read_csv(root_dir / 'data/gesund_bund_icd10.tsv', sep="\t", index_col=False).fillna('')
df

embed

In [ ]:
import requests
def embed(contents: list, host: str="localhost", port: str="8080"):
    """embed texts"""
    response = requests.post(
        f"http://{host}:{port}/embedding",
        headers={"Content-Type": "application/json"},
        json={"content": contents},
    )
    return response.json()

In [ ]:
from tqdm.auto import tqdm
import numpy as np

for key, group in tqdm(df.groupby(np.arange(len(df))//32), total=(len(df)//32 +1)):
    ids = group.id.to_list()
    langs = group.language.to_list()
    texts = []
    for name, row in group.iterrows():
        texts.append((row['title'] + '\n\n' + row['summary']+ '\n\n' + row['text']).strip())
    
    assert len(texts) == len(ids) == len(langs)

    vectors = embed(contents=texts, host='localhost', port='8181')
    assert len(vectors) == len(texts)

    # save in json file
    for v, icd, lang in zip(vectors, ids, langs):
        with output_path.open('a') as f:
            v['icd10'] = icd
            v['language'] = lang
            f.write(json.dumps(v) + '\n')

In [ ]:
import json
output_path = (root_dir / 'results/icd10_emb.jsonl')

vectors = [json.loads(l) for l in output_path.read_text().splitlines()]
vectors = {(v['icd10'], v['language']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

create sqlite database

In [ ]:
from tqdm.auto import tqdm

titles = {'en': [], 'de': []}
texts = {'en': [], 'de': []}
metadatas = {'en': [], 'de': []}
embeds = {'en': [], 'de': []}
for name, row in tqdm(df.iterrows()):
    lang = row['language']
    titles[lang].append(row['title'])
    texts[lang].append((row['summary']+ '\n\n' + row['text']).strip())
    metadatas[lang].append({'icd10': row['id'], 'language': lang})
    embeds[lang].append(vectors[(row['id'], lang)])

In [ ]:
from sqlitevec import SQLiteVec

icd10_table = {}
for lang in ['de', 'en']:
    icd10_table[lang] = SQLiteVec.from_texts(
        titles=titles[lang],
        texts=texts[lang],
        embeds=embeds[lang],
        metadatas=metadatas[lang],
        table=f"icd10_{lang}",
        db_file="icd10.db",
        lang=lang,
    )

In [ ]:
from IPython.display import display, HTML

found = icd10_table['de'].fulltext_search_with_highlight(
    query='title: "Appendizitis"' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['icd10'])
    display(HTML(f['title']))
    print('---')

In [ ]:
found = icd10_table['en'].fulltext_search_with_highlight(
    query='title: "appendicitis"' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['icd10'])
    display(HTML(f['title']))
    print('---')